In [ ]:

import sys
import os
import numpy as np
import mujoco
import time
import mujoco_viewer

sys.path.append(os.path.abspath("../"))

from base_mujoco.utils import *
from base_mujoco.kinematics import *


In [2]:
model_path = "asset/example_scene_ffw_sh5.xml"
model_abs_path = os.path.abspath(model_path)


model = mujoco.MjModel.from_xml_path(model_abs_path)
data = mujoco.MjData(model)

body_names = get_body_names(model, data)
print(body_names)

['world', 'front_object_table', 'camera_ik', 'camera', 'camera2', 'camera3', 'base_link', 'left_wheel_steer', 'left_wheel_drive', 'right_wheel_steer', 'right_wheel_drive', 'rear_wheel_steer', 'rear_wheel_drive', 'lift_link', 'arm_base_link', 'head_link1', 'head_link2', 'arm_l_link1', 'arm_l_link2', 'arm_l_link3', 'arm_l_link4', 'arm_l_link5', 'arm_l_link6', 'arm_l_link7', 'hx5_l_base', 'finger_l_link1', 'finger_l_link2', 'finger_l_link3', 'finger_l_link4', 'finger_l_link5', 'finger_l_link6', 'finger_l_link7', 'finger_l_link8', 'finger_l_link9', 'finger_l_link10', 'finger_l_link11', 'finger_l_link12', 'finger_l_link13', 'finger_l_link14', 'finger_l_link15', 'finger_l_link16', 'finger_l_link17', 'finger_l_link18', 'finger_l_link19', 'finger_l_link20', 'arm_r_link1', 'arm_r_link2', 'arm_r_link3', 'arm_r_link4', 'arm_r_link5', 'arm_r_link6', 'arm_r_link7', 'hx5_r_base', 'finger_r_link1', 'finger_r_link2', 'finger_r_link3', 'finger_r_link4', 'finger_r_link5', 'finger_r_link6', 'finger_r_lin

In [3]:
print(data.qpos)

init_pose = {
    "arm_l_joint4": -1.55,
    "arm_r_joint4": -1.55,
}
for name, val in init_pose.items():
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
    data.qpos[model.jnt_qposadr[jid]] = val

mujoco.mj_forward(model, data)
print(data.qpos)

[-0.3   0.    0.15  1.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.3   0.
  0.75  1.    0.    0.    0.  ]
[-0.3   0.    0.15  1.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.   -1.55  0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   -1.55  0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.3   0.
  0.75  1.    0.    0.    0.  ]


In [4]:
cola_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'obj_coke')
cola_body_pos = data.xpos[cola_body_id]
print(cola_body_pos)

[0.3  0.   0.75]


In [5]:
""" Current POS ROT """
eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
eef_site_pos = data.site_xpos[eef_site_id]
eef_site_rmat = data.site_xmat[eef_site_id]
eef_site_rmat = eef_site_rmat.reshape(3, 3) 
print(f"Current EE position: {eef_site_pos}, Rotation matrix: {eef_site_rmat}")

""" Target POS ROT """
pos_target = np.array([0.3, -0.05, 0.8])
rmat_target = eef_site_rmat.copy()
print(f"Target EE position: {pos_target}, Rotation matrix: {rmat_target}")  

Current EE position: [ 0.23787684 -0.2175      1.27037556], Rotation matrix: [[ 0.02079483 -0.         -0.99978376]
 [ 0.          1.         -0.        ]
 [ 0.99978376  0.          0.02079483]]
Target EE position: [ 0.3  -0.05  0.8 ], Rotation matrix: [[ 0.02079483 -0.         -0.99978376]
 [ 0.          1.         -0.        ]
 [ 0.99978376  0.          0.02079483]]


In [6]:
""" CLIPPING ERROR """
POSITION_CLIPPING = 0.02
ROTATION_CLIPPING = 0.2

""" POS ROT RATIO """
POS_ROT_RATIO = 0.57

In [7]:
def get_ik_error_clipped(
        p_current,
        r_current,
        p_target,
        r_target,
        ):
    pos_error = p_target - p_current
    rmat_error = r_target @ r_current.T
    rotvec_error = rmat2rotvec(rmat_error)* POS_ROT_RATIO
    pos_error_clipped = np.clip(pos_error, -POSITION_CLIPPING, POSITION_CLIPPING)
    rotvec_error_clipped = np.clip(rotvec_error, -ROTATION_CLIPPING, ROTATION_CLIPPING)
    return pos_error_clipped, rotvec_error_clipped

In [8]:
model.nv
for i in range(model.njnt):
    print(model.jnt_type[i], model.joint(i).name)

0 floating_base
3 left_wheel_steer_joint
3 left_wheel_drive_joint
3 right_wheel_steer_joint
3 right_wheel_drive_joint
3 rear_wheel_steer_joint
3 rear_wheel_drive_joint
2 lift_joint
3 head_joint1
3 head_joint2
3 arm_l_joint1
3 arm_l_joint2
3 arm_l_joint3
3 arm_l_joint4
3 arm_l_joint5
3 arm_l_joint6
3 arm_l_joint7
3 finger_l_joint1
3 finger_l_joint2
3 finger_l_joint3
3 finger_l_joint4
3 finger_l_joint5
3 finger_l_joint6
3 finger_l_joint7
3 finger_l_joint8
3 finger_l_joint9
3 finger_l_joint10
3 finger_l_joint11
3 finger_l_joint12
3 finger_l_joint13
3 finger_l_joint14
3 finger_l_joint15
3 finger_l_joint16
3 finger_l_joint17
3 finger_l_joint18
3 finger_l_joint19
3 finger_l_joint20
3 arm_r_joint1
3 arm_r_joint2
3 arm_r_joint3
3 arm_r_joint4
3 arm_r_joint5
3 arm_r_joint6
3 arm_r_joint7
3 finger_r_joint1
3 finger_r_joint2
3 finger_r_joint3
3 finger_r_joint4
3 finger_r_joint5
3 finger_r_joint6
3 finger_r_joint7
3 finger_r_joint8
3 finger_r_joint9
3 finger_r_joint10
3 finger_r_joint11
3 finger_r

In [9]:
for i in range(model.njnt):
      if model.jnt_type[i] == 0:  # free joint만
          print(f"body: {model.body(model.jnt_bodyid[i]).name}, joint: {model.joint(i).name}")

body: base_link, joint: floating_base
body: obj_coke, joint: 


In [10]:
model.njnt

65

In [11]:
joints_use = [                                                                                                                                       
      "lift_joint",                                                                                                                                    
      "arm_r_joint1",                                                                                                                                  
      "arm_r_joint2",                                                                                                                                  
      "arm_r_joint3",                                                                                                                                  
      "arm_r_joint4",                                                                                                                                  
      "arm_r_joint5",
      "arm_r_joint6",
      "arm_r_joint7",
  ]

joints_use_qpos_idxs = [
      model.jnt_qposadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)]
      for name in joints_use
  ]
joints_use_qpos_idxs

[np.int32(13),
 np.int32(43),
 np.int32(44),
 np.int32(45),
 np.int32(46),
 np.int32(47),
 np.int32(48),
 np.int32(49)]

In [ ]:
# viewer = mujoco_viewer.MujocoViewer(model, data)
viewer = mujoco_viewer.MujocoViewer(model, data)
cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, "ikview")                                                                                                                                                
viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
viewer.cam.fixedcamid = cam_id

while viewer.is_alive:
    # get current pos, rotation
    eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
    eef_site_pos = data.site_xpos[eef_site_id]
    eef_site_rmat = data.site_xmat[eef_site_id]
    eef_site_rmat = eef_site_rmat.reshape(3, 3)

    pos_error, rotvec_error = get_ik_error_clipped(
        p_current=eef_site_pos,
        r_current=eef_site_rmat,
        p_target=pos_target,
        r_target=rmat_target
    )
    error = np.concatenate([pos_error, rotvec_error])

    # terminate condition
    if np.linalg.norm(pos_error) < 0.01 and np.linalg.norm(rotvec_error) < 0.01:
        print("Target reached!")

    
    jacobian_p, jacobian_r = get_jacobian(model, data, 'eef_site', type='site', joints_use=joints_use)

    jacobian = np.concat([jacobian_p, jacobian_r], axis=0)

    inversed_jacobian = get_pseudo_inverse(jacobian)

    qpos_error = inversed_jacobian @ error

    data.qpos[joints_use_qpos_idxs] += qpos_error
    mujoco.mj_forward(model, data)

    if viewer.is_alive:
        viewer.add_marker(                                                                                                                                                                                                        
            pos=pos_target,                                                                                                                                                                                                     
            size=np.array([0.01, 0.01, 0.01]),                                                                                                                                                                                 
            rgba=np.array([0.0, 1.0, 0.0, 0.5]),  # 초록색                                                                                                                                                              
            type=mujoco.mjtGeom.mjGEOM_SPHERE,                                                                                                                                                                                    
            label=""                                                                                                                                                                                                              
        )
        viewer.render()
        time.sleep(0.1)

viewer.close()

Target reached!
Target reached!
Target reached!


2026-04-23 23:07:21.794 python[56598:2517416] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target r

: 